# Real photos: glasses, hats, closed eyes and gaze

How the eye and head-covering checks do on real photos, and why their thresholds are where they are. `evaluation.ipynb` does the same on the London Set, whose defects are mostly drawn in; this set has the real thing.

**The photos.** 240 photos from [Open Images](https://storage.googleapis.com/openimages/web/index.html) (Google), each with one large real face: 60 with glasses, 60 with sunglasses, 60 with a hat and 60 with none of these, as Open Images' boxes say. They come from Flickr under CC BY 2.0 (per Google's metadata, which Google didn't verify); `credits.csv` lists author, source and licence, and every photo shown here is credited under it. They are candid photos, not ID photos, so pose and gaze vary far more than in a photo booth.

**The labels.** Claude subagents looked at each photo and recorded what a clerk would object to (`vlm_labels.jsonl`: glasses none/clear/tinted, reflections, frame over the eyes, hair over the eyes, eyes open, gaze, head covering, expression, pose, red eyes, and a note). Photos that aren't a real face (dolls, paintings) or can't be judged are marked `face_judgeable: false` and left out.

Rebuild the set and the results with:

```bash
uv run python scripts/build_openimages_set.py --meta <meta> --out data/openimages/eval
OFIQ_CONFIG_DIR=<OFIQ>/data uv run python scripts/run_eval.py --set data/openimages/eval
```

This notebook shows the photos, so clear its outputs before committing it (the photos stay out of the repository).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))
import score_labels as sl

EVAL = Path(os.environ.get('OPENIMAGES_SET', Path.cwd().parent / 'data' / 'openimages' / 'eval'))

checks = sl.load(EVAL)
labels = pd.read_json(EVAL / 'vlm_labels.jsonl', lines=True).set_index('file')
credits = pd.read_csv(EVAL / 'credits.csv').set_index('file')
judged = labels[labels['face_judgeable'] != False]
rules = pd.read_json(EVAL / 'results.jsonl', lines=True).set_index('file')['rules'].apply(pd.Series)
rules_failed = rules.eq('fail').sum(axis=1)

FINE, PROBLEM, MUTED, OLD = '#2a78d6', '#eb6834', '#52514e', '#b5b4ae'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e6e5e1', 'grid.linewidth': 0.6, 'axes.axisbelow': True})
percent = '{:.0%}'.format


def measure(check, column):
    """One check's measurement per judgeable photo."""
    c = checks[checks['check'] == check].set_index('file')
    return c[column].reindex(judged.index)


def status(check):
    return checks[checks['check'] == check].set_index('file')['status'].reindex(judged.index)


gaze_offset = np.maximum(measure('gaze', 'gaze_horizontal').abs(), measure('gaze', 'gaze_vertical').abs())
eyes_visible = judged['glasses'] != 'tinted'  # behind tinted lenses nobody can judge the eyes
wears_glasses = judged['glasses'] != 'none'
print(f"{len(labels)} photos, {len(judged)} judgeable; glasses: {judged['glasses'].value_counts().to_dict()}")
print(f"{(rules_failed > 0).sum()} of {len(rules)} photos fail at least one rule")

## 1. Each check against the labels

Each check is scored on its own here: almost every photo in this set fails some other rule (a hat, a turned head, a busy background), so the photo's overall verdict says little about any one check. For every label, the photos where it says there is a problem are positives, the rest negatives. `hit` is the share of positives the check warns or fails on (`hit_fail`: fails), `false_alarm` the share of negatives it warns or fails on, and `not_run` the share it skipped. `scored_on` limits a row to the photos where the label means something: `with_glasses` for reflections and frames, `eyes_visible` (no tinted glasses) for eyes, gaze and red eyes.

Some labels have different conventions than the checks: pose and expression are labelled strictly (any speech or slight turn counts), so those false alarms are partly label noise.

In [ ]:
# One row per label, check and the photos it is scored on (a check can be scored on several subsets)
table = sl.score(checks).set_index('scored_on', append=True)
table.style.format({c: percent for c in ('hit', 'hit_fail', 'false_alarm', 'false_fail', 'not_run')}, na_rep='-') \
    .background_gradient(subset=['hit'], cmap='Blues', vmin=0, vmax=1) \
    .background_gradient(subset=['false_alarm'], cmap='Oranges', vmin=0, vmax=1)

## 2. The thresholds

Each panel shows a check's measure for the photos the labels call fine (blue) and for those with the problem (orange), each as a share of its own group, so a small group isn't drowned out. The current warn (dashed) and fail (solid) thresholds are dark grey; where this set moved a threshold, the old ones are light grey.

- **Eyes open:** at the old 80/50, OFIQ failed only 8% of closed eyes; most closed eyes score 50-90.
- **Eyes visible:** clear glasses often score 50-90 (the frame and reflections lower it), so the old warn at 90 flagged most of them. Tinted lenses score near 0.

In [ ]:
# (title, values, problem mask, photos to show, warn, fail, old (warn, fail) or None, fine label, problem label)
PANELS = [
    ('Eyes open (OFIQ)', measure('ofiq_eyes_open', 'ofiq_eyes_open_quality'), judged['eyes_open'] == False,
     eyes_visible, 95, 70, (80, 50), 'eyes open', 'eyes not open'),
    ('Eyes visible (OFIQ)', measure('ofiq_eyes_visible', 'ofiq_eyes_visible_quality'), judged['glasses'] == 'tinted',
     wears_glasses, 60, 25, (90, 50), 'clear glasses', 'tinted glasses'),
    ('Under-eye / cheek lightness', measure('glasses_tint', 'lens_brightness_ratio'), judged['glasses'] == 'tinted',
     wears_glasses, 0.55, 0.45, None, 'clear glasses', 'tinted glasses'),
    ('No head covering (OFIQ)', measure('ofiq_head_covering', 'ofiq_no_head_coverings_quality'), judged['head_covering'] != 'none',
     judged.index == judged.index, 90, 50, None, 'no head covering', 'head covering'),
    ('Gaze offset (share of eye width)', gaze_offset, judged['gaze_at_camera'] == False,
     eyes_visible, 0.08, 0.15, None, 'looks at the camera', 'looks away'),
    ('Glare share of the lenses', measure('glasses_glare', 'lens_glare_share'), judged['glasses_glare'] == True,
     wears_glasses, 0.01, 0.04, None, 'no reflections', 'reflections'),
]


def threshold_line(ax, value, kind, color):
    ax.axvline(value, color=color, linestyle='--' if 'warn' in kind else '-', linewidth=1)
    ax.text(value, 0.98, kind, transform=ax.get_xaxis_transform(), rotation=90, color=color, fontsize=7,
            ha='right', va='top')


fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
for ax, (title, values, problem, subset, warn, fail, old, fine_label, problem_label) in zip(axes.flat, PANELS):
    values = values[subset].astype(float)
    problem = problem[subset]
    bins = np.linspace(values.min(), values.max(), 30)
    for mask, color, label in ((~problem, FINE, fine_label), (problem, PROBLEM, problem_label)):
        v = values[mask].dropna()
        ax.hist(v, bins=bins, weights=np.full(len(v), 1 / max(1, len(v))), color=color, alpha=0.7,
                label=f'{label} (n={len(v)})', edgecolor='white', linewidth=0.5)
    if old:
        threshold_line(ax, old[0], 'old warn', OLD)
        threshold_line(ax, old[1], 'old fail', OLD)
    threshold_line(ax, warn, 'warn', MUTED)
    threshold_line(ax, fail, 'fail', MUTED)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('share of group', fontsize=8)
    ax.legend(fontsize=8, frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=2)
plt.show()

## 3. The photos behind the numbers

For each check, a few photos it caught, missed, and wrongly flagged (picked at random). Above each photo is **that check's** status and measure, not the photo's verdict; the line below gives the verdict of the whole pipeline (how many rules the photo fails) and the Flickr author. A wrong flag on a photo that fails other rules anyway costs nothing; what costs a retake is a wrong flag on an otherwise good photo, which the London Set measures (`evaluation.ipynb`, section 2).

In [ ]:
def verdict(file):
    failed = rules_failed.get(file, 0)
    return f'photo fails {failed} rule' + ('s' if failed != 1 else '') if failed else 'photo fails no rule'


def show(files, title, column_values, n=7, seed=0):
    files = list(pd.Index(files).to_series().sample(min(n, len(files)), random_state=seed)) if len(files) else []
    fig, axes = plt.subplots(1, n, figsize=(1.9 * n, 3.2))
    fig.subplots_adjust(left=0.005, right=0.995, top=0.8, bottom=0.13, wspace=0.06)
    fig.suptitle(f'{title} ({len(files)} shown)', x=0.005, ha='left', fontsize=10)
    for ax in axes:
        ax.axis('off')
    for ax, file in zip(axes, files):
        image = Image.open(EVAL / file)
        image.thumbnail((300, 300))
        ax.imshow(image)
        value = column_values.get(file, np.nan)
        measured = f' · {value:.3g}' if pd.notna(value) else ''
        ax.set_title(f'{check_name}: {status_of[file]}{measured}', fontsize=8)
        ax.text(0.5, -0.03, verdict(file), transform=ax.transAxes, ha='center', va='top', fontsize=7.5)
        ax.text(0.5, -0.11, f"{file[3:11]} · {str(credits.loc[file, 'author'])[:22]}", transform=ax.transAxes,
                ha='center', va='top', fontsize=6.5, color=MUTED)
    plt.show()


def examples(check, values, problem, subset, n=7):
    """Caught, missed and false alarms of one check."""
    global status_of, check_name
    status_of, check_name = status(check), check
    flagged = status_of.isin(['warn', 'fail'])
    ran = status_of.isin(['pass', 'warn', 'fail'])
    pick = lambda mask: judged.index[(mask & subset & ran).to_numpy()]
    show(pick(problem & flagged), 'Caught', values, n)
    show(pick(problem & ~flagged), 'Missed', values, n)
    show(pick(~problem & flagged), 'False alarms', values, n)

### Eyes open (`ofiq_eyes_open`, warn below 95, fail below 70)

In [ ]:
examples('ofiq_eyes_open', measure('ofiq_eyes_open', 'ofiq_eyes_open_quality'), judged['eyes_open'] == False, eyes_visible)

### Tinted glasses (`glasses_tint`, warn below 0.55, fail below 0.45)

In [ ]:
examples('glasses_tint', measure('glasses_tint', 'lens_brightness_ratio'), judged['glasses'] == 'tinted', wears_glasses)

### Eyes visible (`ofiq_eyes_visible`, warn below 60, fail below 25), on glasses wearers

In [ ]:
examples('ofiq_eyes_visible', measure('ofiq_eyes_visible', 'ofiq_eyes_visible_quality'), judged['glasses'] == 'tinted', wears_glasses)

### Head covering (`ofiq_head_covering`, warn below 90, fail below 50)

In [ ]:
examples('ofiq_head_covering', measure('ofiq_head_covering', 'ofiq_no_head_coverings_quality'), judged['head_covering'] != 'none', judged['glasses'] == judged['glasses'])

### Gaze (`gaze`, warn above 0.08, fail above 0.15)

In [ ]:
examples('gaze', gaze_offset, judged['gaze_at_camera'] == False, eyes_visible)

### Reflections on clear glasses (`glasses_glare`, warn above 0.01, fail above 0.04); sunglasses fail anyway

In [ ]:
examples('glasses_glare', measure('glasses_glare', 'lens_glare_share'), judged['glasses_glare'] == True, judged['glasses'] == 'clear')

The glare check only sees blown-out white spots; most real reflections are soft sheens or mirrored windows, which is why it misses 9 in 10. The rule `glasses_no_reflections` is therefore only partly covered.

## 4. CLIP and SigLIP as a second opinion

`scripts/clip_zero_shot.py` scored every photo zero-shot (problem prompts minus fine prompts, on the eye region for eye questions) with four models: SigLIP ViT-B-16 (800 MB), LAION CLIP ViT-B-32 (600 MB), OpenAI CLIP ViT-L-14 (1.6 GB) and SigLIP SO400M (3.3 GB). They need PyTorch, so the script runs in its own environment; this cell only reads the saved scores. The prompts were written for SigLIP B-16 and not tuned per model. AUC is the chance that a random photo with the problem scores higher than a random one without it (0.5 is chance, 1.0 perfect).

In [ ]:
def auc(score, positive):
    """Chance that a random positive scores higher than a random negative (ties count half)."""
    ok = score.notna() & positive.notna()
    score, positive = score[ok].astype(float), positive[ok].astype(bool)
    ranks = score.rank()
    n_pos, n_neg = positive.sum(), (~positive).sum()
    return (ranks[positive].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)


MODELS = {'SigLIP B-16': 'siglip_scores.csv', 'CLIP B-32': 'clip_b32_scores.csv',
          'CLIP L-14': 'clip_l14_scores.csv', 'SigLIP SO400M': 'siglip_so400m_scores.csv'}
models = {}
for model, filename in MODELS.items():
    if (EVAL / filename).exists():
        s = pd.read_csv(EVAL / filename)
        models[model] = s[s['set'] == 'openimages'].set_index('file').reindex(judged.index)
# (question, problem mask, photos, our measure (higher = problem), its name)
QUESTIONS = [
    ('glare', judged['glasses_glare'] == True, wears_glasses, measure('glasses_glare', 'lens_glare_share'), 'glasses_glare'),
    ('tinted', judged['glasses'] == 'tinted', judged['glasses'] == judged['glasses'],
     100 - measure('ofiq_eyes_visible', 'ofiq_eyes_visible_quality'), 'ofiq_eyes_visible'),
    ('hat', judged['head_covering'] != 'none', judged['glasses'] == judged['glasses'],
     100 - measure('ofiq_head_covering', 'ofiq_no_head_coverings_quality'), 'ofiq_head_covering'),
    ('eyes_closed', judged['eyes_open'] == False, eyes_visible, 100 - measure('ofiq_eyes_open', 'ofiq_eyes_open_quality'), 'ofiq_eyes_open'),
    ('gaze_aside', judged['gaze_at_camera'] == False, eyes_visible, gaze_offset, 'gaze'),
]
table = pd.DataFrame([
    {'question': q, 'positives': int(problem[subset].sum()), 'photos': int(subset.sum()), 'our check': name,
     'ours': auc(ours[subset], problem[subset]), **{m: auc(s.loc[subset, q], problem[subset]) for m, s in models.items()}}
    for q, problem, subset, ours, name in QUESTIONS
]).set_index('question')
aucs = ['ours', *models]
table.style.format({c: '{:.2f}' for c in aucs}).background_gradient(subset=aucs, cmap='Blues', vmin=0.5, vmax=1, axis=None)

- **Reflections on glasses:** every model ranks them better than the pixel check, the two large ones most. Much of that is sunglasses, though, which fail anyway: on clear glasses alone (19 reflections), SigLIP SO400M catches about a third, and together with the pixel check 8 of 19 instead of 2, with false warnings on 8 instead of 3 of 48 clear-glasses photos.
- **Gaze:** only SigLIP SO400M beats the iris-offset check, on both halves of the set (`calibrate` and `test`). Both CLIP models do badly here.
- **Closed eyes:** CLIP L-14 scores highest, but only on one half; with 13 closed-eye photos per half, that isn't settled.
- **Tinted glasses and hats:** every model and our checks are close to the top.

So the pipeline runs SigLIP SO400M as an optional, warn-only second opinion for reflections (`siglip_glasses_glare`) and gaze (`siglip_gaze`), switched on with `SIGLIP_ENABLED=true`. Both warn above the score that 80% of fine photos on the `calibrate` half stay under (-1.12 and -0.33). No London studio photo goes over either.

## 5. What this doesn't show

- **Candid photos, not ID photos.** People talk, laugh and look around, and the light is anything. Real submissions will sit between this set and the London Set.
- **The labels are a model's judgement.** The labelling subagents disagreed on borderline calls: downcast eyes (closed or not?), one hidden eye, and whether speaking counts as a non-neutral expression.
- **Too few cases** of hair over the eyes (5) and red eyes (2) to say anything.
- **Old thresholds aren't re-run here.** The light grey lines in section 2 show where they were; the numbers in `PASSPORT_VALIDATOR.md` compare before and after.